# Mini_Assignment_1_Shradha_Bavalatti

In [10]:
# Install PySpark
from google.colab import files
uploaded = files.upload()
import pandas as pd
df_pd = pd.read_csv("Lung Cancer.csv")
df_pd.head()

Saving Lung Cancer.csv to Lung Cancer (1).csv


,id,age,gender,country,diagnosis_date,cancer_stage,family_history,smoking_status,bmi,cholesterol_level,hypertension,asthma,cirrhosis,other_cancer,treatment_type,end_treatment_date,survived
0,1,64.0,Male,Sweden,2016-04-05,Stage I,Yes,Passive Smoker,29.4,199,0,0,1,0,Chemotherapy,2017-09-10,0
1,2,50.0,Female,Netherlands,2023-04-20,Stage III,Yes,Passive Smoker,41.2,280,1,1,0,0,Surgery,2024-06-17,1
2,3,65.0,Female,Hungary,2023-04-05,Stage III,Yes,Former Smoker,44.0,268,1,1,0,0,Combined,2024-04-09,0
3,4,51.0,Female,Belgium,2016-02-05,Stage I,No,Passive Smoker,43.0,241,1,1,0,0,Chemotherapy,2017-04-23,0
4,5,37.0,Male,Luxembourg,2023-11-29,Stage I,No,Passive Smoker,19.7,178,0,0,0,0,Combined,2025-01-08,0


In [11]:
!pip install pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
spark = SparkSession.builder.getOrCreate()
df = spark.createDataFrame(df_pd)
df.show(5)
df.printSchema()

+---+----+------+-----------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
| id| age|gender|    country|diagnosis_date|cancer_stage|family_history|smoking_status| bmi|cholesterol_level|hypertension|asthma|cirrhosis|other_cancer|treatment_type|end_treatment_date|survived|
+---+----+------+-----------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
|  1|64.0|  Male|     Sweden|    2016-04-05|     Stage I|           Yes|Passive Smoker|29.4|              199|           0|     0|        1|           0|  Chemotherapy|        2017-09-10|       0|
|  2|50.0|Female|Netherlands|    2023-04-20|   Stage III|           Yes|Passive Smoker|41.2|              280|           1|     1|        0|           0|       Surgery|        2024-06-17|       1|
|  3|65.0|Femal

## Task 1 – Write a function that removes duplicate rows, ensures correct data types for numerical and date columns and converts all ‘yes’/ ‘no’ type fields into 1/0 format.  

In [14]:
from pyspark.sql.functions import *
from pyspark.sql.types import StringType
def clean_data(df):
    # 1. Detect yes/no columns
    bool_cols = []
    for c in df.columns:
        if df.schema[c].dataType == StringType():
            vals = [str(v).lower() for v in df.select(c).distinct().toPandas()[c]]
            if set(vals).issubset({"yes", "no"}):
                bool_cols.append(c)

    # 2. Convert yes/no → 1/0
    for c in bool_cols:
        df = df.withColumn(c, when(lower(col(c)) == "yes", 1).otherwise(0))

    # 3. Remove duplicate rows
    df = df.dropDuplicates()

    return df


# Run Task 1
df_clean = clean_data(df)

# Display the cleaned output
df_clean.show(10)
df_clean.printSchema()

+----+----+------+--------------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
|  id| age|gender|       country|diagnosis_date|cancer_stage|family_history|smoking_status| bmi|cholesterol_level|hypertension|asthma|cirrhosis|other_cancer|treatment_type|end_treatment_date|survived|
+----+----+------+--------------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
| 276|55.0|  Male|Czech Republic|    2014-10-13|    Stage IV|             0| Former Smoker|19.1|              170|           1|     0|        1|           0|  Chemotherapy|        2015-12-25|       0|
| 361|55.0|Female|       Germany|    2018-09-18|   Stage III|             1| Former Smoker|36.7|              243|           1|     0|        0|           0|  Chemotherapy|        2019-09-12|     

## Task 2 – Write a function that adds a new column, treatment_duration_days, which calculates the number of days between the diagnosis and the end of treatment. Then, return the average treatment duration for each treatment type.  

In [16]:
from pyspark.sql.functions import *
def avg_treatment_duration(df):
    df = df.withColumn(
        "treatment_duration_days",
        datediff(col("end_treatment_date"), col("diagnosis_date"))
    )

    result = df.groupBy("treatment_type").agg(
        avg("treatment_duration_days").alias("avg_duration_days")
    )

    return result


# Run Task 2
result_task2 = avg_treatment_duration(df_clean)

# Task 2 output
result_task2.show()

+--------------+------------------+
|treatment_type| avg_duration_days|
+--------------+------------------+
|     Radiation|458.40320462900917|
|  Chemotherapy|458.39540091909953|
|      Combined| 457.8152186120058|
|       Surgery|457.73744630723684|
+--------------+------------------+



## Task 3 – Write a function that returns the smoking_status group with the highest survival rate.  

In [17]:
from pyspark.sql.functions import *
def highest_survival_rate(df):
    result = df.groupBy("smoking_status").agg(
        avg(col("survived").cast("int")).alias("survival_rate")
    )

    # Return the smoking status with the highest survival rate
    return result.orderBy(col("survival_rate").desc()).limit(1)


#  Run Task 3
result_task3 = highest_survival_rate(df_clean)

#  result
result_task3.show()

+--------------+-------------------+
|smoking_status|      survival_rate|
+--------------+-------------------+
|  Never Smoked|0.22091034383684025|
+--------------+-------------------+



## Task 4 –  Write a function that returns the top three countries with the highest percentage of patients diagnosed in Stage IV.  

In [18]:
from pyspark.sql.functions import *
def top_countries_stage4(df):
    # Create a binary indicator: 1 if Stage IV, else 0
    df_stage = df.withColumn(
        "is_stage4",
        (col("cancer_stage") == "Stage IV").cast("int")
    )

    # Calculate Stage IV percentage per country
    result = df_stage.groupBy("country").agg(
        avg("is_stage4").alias("stage4_percentage")
    )

    # Return top 3 countries
    return result.orderBy(col("stage4_percentage").desc()).limit(3)


# Run Task 4
result_task4 = top_countries_stage4(df_clean)

# result
result_task4.show()

+--------------+-------------------+
|       country|  stage4_percentage|
+--------------+-------------------+
|        Greece| 0.2550223889628464|
|       Croatia| 0.2542700223308588|
|Czech Republic|0.25291166185190816|
+--------------+-------------------+



## Task 5 – Write a function that filters patients who:  

Are male  
Diagnosed in Stage III or IV  
Have a family history of cancer  
Are current smokers  
Have a BMI > 30  
Survived

In [19]:
from pyspark.sql.functions import *
def filter_group_stats(df):
    # Apply all filters
    filtered = df \
        .filter(col("gender") == "Male") \
        .filter(col("cancer_stage").isin("Stage III", "Stage IV")) \
        .filter(col("family_history") == 1) \
        .filter(col("smoking_status") == "Current Smoker") \
        .filter(col("bmi") > 30) \
        .filter(col("survived") == 1)

    # Calculate average age
    avg_age = filtered.agg(avg("age")).first()[0]

    # Calculate hypertension percentage
    hypertension_pct = filtered.agg(
        avg(col("hypertension").cast("int"))
    ).first()[0]

    return avg_age, hypertension_pct


# Run Task 5
avg_age, hypertension_pct = filter_group_stats(df_clean)

# results
print("Average Age:", avg_age)
print("Hypertension Percentage:", hypertension_pct)

Average Age: 55.179398872886665
Hypertension Percentage: 0.7476518472135254
